# Week 1 Data Exploration Template

这个 notebook 用来检查 Week 1 生成的 `queries.jsonl`、`passages.jsonl`、`qrels.jsonl` 是否可信。

请运行每个代码块，并在标有 **TODO** 的地方填写你的观察和结论。

本 notebook 的目标不是训练模型，而是回答：

1. 数据是否能正常读取？
2. qrels 是否都能映射到真实 passage？
3. 是否有空 passage、重复 ID、query 没有 gold evidence 等问题？
4. 人工检查若干样本后，gold evidence 是否合理？
5. 是否可以进入 Week 2 的 BM25 baseline？

## 1. 配置数据路径

如果你的 processed data 不在 `data/processed`，请修改下面的 `DATA_DIR`。

In [1]:
from pathlib import Path
import json
import random
from collections import Counter, defaultdict

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# TODO: 如果路径不对，改这里
DATA_DIR = PROJECT_ROOT / "data" / "processed"

QUERIES_PATH = DATA_DIR / "queries.jsonl"
PASSAGES_PATH = DATA_DIR / "passages.jsonl"
QRELS_PATH = DATA_DIR / "qrels.jsonl"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("queries exists:", QUERIES_PATH.exists())
print("passages exists:", PASSAGES_PATH.exists())
print("qrels exists:", QRELS_PATH.exists())

PROJECT_ROOT: /home/haris/hotpot-evidence-retrieval
DATA_DIR: /home/haris/hotpot-evidence-retrieval/data/processed
queries exists: True
passages exists: True
qrels exists: True


## 2. 读取 JSONL 文件

In [2]:
def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}:{line_no}") from exc
    return records

queries = read_jsonl(QUERIES_PATH)
passages = read_jsonl(PASSAGES_PATH)
qrels = read_jsonl(QRELS_PATH)

print("num queries:", len(queries))
print("num passages:", len(passages))
print("num qrels:", len(qrels))

num queries: 100
num passages: 4085
num qrels: 243


### 文件读取结果观察

- 本次处理了 100 个 queries
- 生成了多少 4086 个 passages (更改脚本后为 4085)
- 生成了多少 243 个 qrels
- 这些数量和 `latest_run.txt` 一致

## 3. 建立索引

In [3]:
query_by_id = {item["query_id"]: item for item in queries}
passage_by_id = {item["passage_id"]: item for item in passages}

qrels_by_query = defaultdict(list)
for qrel in qrels:
    qrels_by_query[qrel["query_id"]].append(qrel["passage_id"])

print("unique query ids:", len(query_by_id))
print("unique passage ids:", len(passage_by_id))
print("queries with qrels:", len(qrels_by_query))

unique query ids: 100
unique passage ids: 4085
queries with qrels: 100


## 4. 基础统计

请补全或运行下面的统计。

In [4]:
query_type_counts = Counter(item.get("type", "unknown") for item in queries)
passage_lengths = [len(item.get("passage_text", "").split()) for item in passages]
qrels_per_query = [len(qrels_by_query.get(item["query_id"], [])) for item in queries]

summary = {
    "num_queries": len(queries),
    "num_passages": len(passages),
    "num_qrels": len(qrels),
    "avg_qrels_per_query": sum(qrels_per_query) / len(qrels_per_query) if qrels_per_query else 0,
    "min_qrels_per_query": min(qrels_per_query) if qrels_per_query else 0,
    "max_qrels_per_query": max(qrels_per_query) if qrels_per_query else 0,
    "avg_passage_length_words": sum(passage_lengths) / len(passage_lengths) if passage_lengths else 0,
    "min_passage_length_words": min(passage_lengths) if passage_lengths else 0,
    "max_passage_length_words": max(passage_lengths) if passage_lengths else 0,
    "query_type_counts": dict(query_type_counts),
}

summary

{'num_queries': 100,
 'num_passages': 4085,
 'num_qrels': 243,
 'avg_qrels_per_query': 2.43,
 'min_qrels_per_query': 2,
 'max_qrels_per_query': 5,
 'avg_passage_length_words': 22.130722154222767,
 'min_passage_length_words': 1,
 'max_passage_length_words': 178,
 'query_type_counts': {'comparison': 21, 'bridge': 79}}

### 基础统计结论

- 数据包含 79 个 bridge 问题和 21 个 comparison 问题
- 每个 query 平均有 2.43 条 gold passages， 最多 5 条， 最少 2 条。
- passage 平均长度约为 22.13 个词
- 最短 passage 长度为 0，说明数据中可能存在空 passage，需要进一步检查（检查并更改后最短 passage 长度为 1）

## 5. 数据一致性检查

这一部分检查 Week 1 最关键的问题：qrels 是否能正确指向 queries 和 passages。

In [5]:
duplicate_query_ids = [qid for qid, count in Counter(item["query_id"] for item in queries).items() if count > 1]
duplicate_passage_ids = [pid for pid, count in Counter(item["passage_id"] for item in passages).items() if count > 1]

qrels_missing_query = [qrel for qrel in qrels if qrel["query_id"] not in query_by_id]
qrels_missing_passage = [qrel for qrel in qrels if qrel["passage_id"] not in passage_by_id]
queries_without_qrels = [item["query_id"] for item in queries if item["query_id"] not in qrels_by_query]
empty_passages = [item for item in passages if not item.get("passage_text", "").strip()]

checks = {
    "duplicate_query_ids": len(duplicate_query_ids),
    "duplicate_passage_ids": len(duplicate_passage_ids),
    "qrels_missing_query": len(qrels_missing_query),
    "qrels_missing_passage": len(qrels_missing_passage),
    "queries_without_qrels": len(queries_without_qrels),
    "empty_passages": len(empty_passages),
}

checks

{'duplicate_query_ids': 0,
 'duplicate_passage_ids': 0,
 'qrels_missing_query': 0,
 'qrels_missing_passage': 0,
 'queries_without_qrels': 0,
 'empty_passages': 0}

In [6]:
# 如果下面有输出，需要在结论里说明，并考虑回到 build_hotpotqa_dataset.py 修复。
print("qrels_missing_passage sample:")
for item in qrels_missing_passage[:5]:
    print(item)

print("\nqueries_without_qrels sample:")
for item in queries_without_qrels[:5]:
    print(item)

print("\nempty_passages sample:")
for item in empty_passages[:5]:
    print(item)

qrels_missing_passage sample:

queries_without_qrels sample:

empty_passages sample:


### 数据一致性检查结论

- query ID 和 passage ID 均无重复：
  - `duplicate_query_ids = 0`
  - `duplicate_passage_ids = 0`

- 所有 qrels 都能映射到有效的 query 和 passage：
  - `qrels_missing_query = 0`
  - `qrels_missing_passage = 0`

- 所有 queries 都至少有一条 qrel：
  - `queries_without_qrels = 0`

- 发现 1 个空 passage：
  - `1989 Houston Oilers season::10`
  - 后续处理：先确认该 passage 是否属于 gold evidence；如果不是，应在数据生成阶段过滤空文本。  

- 当前 qrels 在结构上基本可信，但还需要人工检查部分 query 与对应 gold passages，确认 passage 内容确实能够支持答案。

In [7]:
empty_pid = "1989 Houston Oilers season::10"

[qrel for qrel in qrels if qrel["passage_id"] == empty_pid]

[]


- 这说明这个空 passage 没有出现在 qrels 中，它不是 gold evidence，可以安全过滤。
- 更改原始脚本并重新运行后 `empty_passages = 0`

## 6. 人工检查 gold evidence

请至少人工检查 20 条 query。下面提供一个展示函数。

In [8]:
def show_query_evidence(query_id):
    query = query_by_id[query_id]
    gold_ids = qrels_by_query.get(query_id, [])

    print("=" * 100)
    print("query_id:", query_id)
    print("type:", query.get("type"))
    print("question:", query.get("query"))
    print("answer:", query.get("answer"))
    print("gold passage ids:", gold_ids)
    print("-" * 100)

    for pid in gold_ids:
        passage = passage_by_id.get(pid)
        if passage is None:
            print("MISSING PASSAGE:", pid)
            continue
        print(f"[{pid}]")
        print(passage.get("passage_text"))
        print()

# TODO: 你可以修改 random seed 或 sample 数量。
random.seed(0)
sample_query_ids = random.sample([item["query_id"] for item in queries], k=min(20, len(queries)))

for qid in sample_query_ids:
    show_query_evidence(qid)

query_id: q_000050
type: bridge
question: Which British first-generation jet-powered medium bomber was used in the South West Pacific theatre of World War II?
answer: English Electric Canberra
gold passage ids: ['No. 2 Squadron RAAF::5', 'English Electric Canberra::0']
----------------------------------------------------------------------------------------------------
[No. 2 Squadron RAAF::5]
 It saw action as a bomber unit in the South West Pacific theatre of World War II and, equipped with English Electric Canberra jets, in the Malayan Emergency and the Vietnam War.

[English Electric Canberra::0]
The English Electric Canberra is a British first-generation jet-powered medium bomber that was manufactured during the 1950s.

query_id: q_000098
type: comparison
question: Which filmmaker was known for animation, Lev Yilmaz or Pamela B. Green?
answer: Levni Yilmaz
gold passage ids: ['Lev Yilmaz::0', 'Pamela B. Green::0']
---------------------------------------------------------------------

### 人工检查记录

1. 随机检查了 **20 条** query。

2. 大多数 gold passages 都能够支持答案。  
   - bridge 问题通常由一条 passage 找到中间实体，再由另一条 passage 得到最终答案。  
   - comparison 问题通常分别提供两个对象的信息，再进行比较。

3. 没有发现明显完全不相关的 qrels。  
   - 但 `q_000054` 的证据关系不够直接：passages 分别说明电影拍摄于 Rome，以及 Franco Corelli 被称为 “Prince of tenors”，但没有明确说明他出演了该电影。

4. 在这 20 条 gold evidence 中没有发现空 passage。大部分 passage 长度正常，但有少量文本格式异常，例如多余符号、空括号或名称格式不完整：
   - `Giuseppe Verdi::0` 中出现 `Giuseppe Fortunino Francesco Verdi (] ;`
   - `Bordan Tkachuk::0` 中出现 `Bordan Tkachuk ( )`

   这些格式问题目前不影响 qrel 映射。

5. 映射正确的例子：
   - `q_000006`：第一条 passage 说明 `2014 S/S` 是 WINNER 的首张专辑，第二条说明 WINNER 由 YG Entertainment 组建，因此答案为 `YG Entertainment`。
   - `q_000034`：一条 passage 说明 *Freakonomics* 是美国纪录片，另一条说明 *In the Realm of the Hackers* 是澳大利亚纪录片，因此答案为 `no`。
   - `q_000075`：一条 passage 说明 Robert Suettinger 是 Bill Clinton 任内的情报官员，另外两条说明 Bill Clinton 的全名及其曾任 Arkansas 州长，因此答案为 `William Jefferson Clinton`。

## 7. 按问题类型检查

HotpotQA 通常包含 `bridge` 和 `comparison` 两类问题。请分别看一些例子。

In [10]:
queries_by_type = defaultdict(list)
for item in queries:
    queries_by_type[item.get("type", "unknown")].append(item["query_id"])

for query_type, ids in queries_by_type.items():
    print("\n" + "#" * 100)
    print("TYPE:", query_type, "COUNT:", len(ids))
    print("#" * 100)
    for qid in ids[:2]:
        show_query_evidence(qid)


####################################################################################################
TYPE: comparison COUNT: 21
####################################################################################################
query_id: q_000001
type: comparison
question: Were Scott Derrickson and Ed Wood of the same nationality?
answer: yes
gold passage ids: ['Scott Derrickson::0', 'Ed Wood::0']
----------------------------------------------------------------------------------------------------
[Scott Derrickson::0]
Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.

[Ed Wood::0]
Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.

query_id: q_000004
type: comparison
question: Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?
answer: no
gold passage ids: ['Laleli Mosque::0', 'Esma Sultan Mansion::0']
----------------------------------------

### bridge / comparison 的观察

- Bridge query 的 gold evidence 通常呈现为一条由两条及以上 passages 连接的多跳证据链，逻辑为 “问题实体 → 中间实体 → 答案”。
- comparison query 的 gold evidence 通常呈现为分别涵盖两个对象的属性的两条 passages。
- Bridge query 更可能需要图结构辅助，因为它需要沿着 “问题实体 → 中间实体 → 答案实体” 的关系进行多跳检索。

### 我的最终结论

1. 本次处理了 `100` 个 queries，生成 `4085` 个 passages 和 `243` 个 qrels。
2. qrels_missing_passage = `0`，queries_without_qrels = `0`，empty_passages = `0`。
3. 我人工检查了 `20` 条 query，gold evidence 映射基本合理。